# Validating LLM Labels

This notebook loads the repo-specific LLM outputs for both `sigma` and `ssc` and recomputes structural comparison signals from the original source files.

Original sources used here:

- `data_prep/align_data/all_steps_{repo}.jsonl`
- `analysis/scripts/.cache/struct_ops_{repo}.pkl`
- fallback to `data_prep/ir_data/pgir_{repo}_nonempty.jsonl` when the cache is missing

The mismatch checks use the updated LLM fields and these PG-IR support rules:

- `predicate_modified_present` is supported by `PRED_UPDATE`
- `predicate_added` is supported by `AND_ADD`, `OR_ADD`, and `BRANCH_ADD`
- `predicate_removed` is supported by `AND_REMOVE`, `OR_REMOVE`, and `BRANCH_REMOVE`


In [1]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data_prep").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find repo root containing 'data_prep'")

sys.path.insert(0, str(ROOT / "data_prep" / "llm_src"))
sys.path.insert(0, str(ROOT / "data_prep" / "align_src"))

from structural_test_labels import load_nonnoop_steps, run_detection_cached

REPO_RESULT_PATHS = {
    "sigma": ROOT / "data_prep" / "llm_data" / "sigma" / "pair_changed_results.jsonl",
    "ssc": ROOT / "data_prep" / "llm_data" / "ssc" / "pair_changed_results.jsonl",
}

manifest_rows_by_repo = {}
for repo, manifest_path in REPO_RESULT_PATHS.items():
    with manifest_path.open(encoding="utf-8") as f:
        manifest_rows_by_repo[repo] = [json.loads(line) for line in f if line.strip()]


def display_table(rows: list[dict], sort_by: str | None = None, descending: bool = True):
    rows = list(rows)
    if sort_by is not None:
        rows = sorted(rows, key=lambda row: row[sort_by], reverse=descending)
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        pprint(rows)
    return rows

for repo, rows in manifest_rows_by_repo.items():
    repo_counts = dict(sorted(Counter(row["repo"] for row in rows).items()))
    print(f"{repo}: manifest rows = {len(rows)}")
    print(f"{repo}: repos in file = {repo_counts}")


sigma: manifest rows = 7958
sigma: repos in file = {'sigma': 7958}
ssc: manifest rows = 4412
ssc: repos in file = {'ssc': 4412}


## Recompute Structural Signals From Original Files

In [2]:
llm_field_support = {
    "predicate_modified_present": {
        "PRED_UPDATE", 
        "PRED_SCOPE_REASSIGN", "PRED_CONTEXT_RELABEL",
    },
    "predicate_added": {
        "AND_ADD", "OR_ADD", "BRANCH_ADD",
        "PRED_SCOPE_REASSIGN", "PRED_CONTEXT_RELABEL",
    },
    "predicate_removed": {
        "AND_REMOVE", "OR_REMOVE", "BRANCH_REMOVE",
        "PRED_SCOPE_REASSIGN", "PRED_CONTEXT_RELABEL",
    },
}

tracked_ops = sorted({op for ops in llm_field_support.values() for op in ops})

def analyze_repo(repo: str, manifest_rows: list[dict]) -> dict:
    steps = load_nonnoop_steps(repo)
    step_by_key = {
        (repo, step["lineage_id"], step["version_a"], step["version_b"]): step
        for step in steps
    }

    det_by_key = {}
    for det in run_detection_cached(repo, steps):
        if det is None or "error" in det:
            continue
        key = (repo, det["lineage_id"], det["version_a"], det["version_b"])
        det_by_key[key] = det

    joined_rows = []
    missing_keys = []

    for row in manifest_rows:
        key = (repo, row["lineage_id"], row["version_a"], row["version_b"])
        step = step_by_key.get(key)
        det = det_by_key.get(key)
        if step is None or det is None:
            missing_keys.append(key)
            continue

        active_ops = sorted(
            op for op, is_on in (det.get("ops") or {}).items()
            if is_on and op in tracked_ops
        )
        supported_llm_fields = sorted(
            field for field, supporting_ops in llm_field_support.items()
            if set(active_ops) & supporting_ops
        )

        joined_rows.append({
            **row,
            "step_row": step,
            "det_row": det,
            "active_ops": active_ops,
            "supported_llm_fields": supported_llm_fields,
        })

    direction_rationale_counts = Counter(
        (row["llm_result"]["match_set_direction"], row["llm_result"]["rationale_label"])
        for row in joined_rows
    )

    field_mismatch_counts = {}
    field_mismatches = []
    for llm_field, supporting_ops in llm_field_support.items():
        mismatches = []
        for row in joined_rows:
            llm_value = bool(row["llm_result"].get(llm_field, False))
            active_ops = set(row["active_ops"])
            supporting_evidence = bool(active_ops & supporting_ops)
            if llm_value and not supporting_evidence:
                mismatches.append({
                    "repo": row["repo"],
                    "lineage_id": row["lineage_id"],
                    "version_a": row["version_a"],
                    "version_b": row["version_b"],
                    "field": llm_field,
                    "llm_value": llm_value,
                    "supporting_evidence": supporting_evidence,
                    "active_ops": row["active_ops"],
                    "supporting_ops": sorted(supporting_ops),
                    "supported_llm_fields": row["supported_llm_fields"],
                    "summary": row["llm_result"]["summary"],
                    "rule_canonical": row["rule_canonical"],
                })
        field_mismatch_counts[llm_field] = len(mismatches)
        field_mismatches.extend(mismatches)

    recomputed_field_counts = Counter()
    for row in joined_rows:
        recomputed_field_counts.update(row["supported_llm_fields"])

    return {
        "manifest_rows": manifest_rows,
        "joined_rows": joined_rows,
        "missing_keys": missing_keys,
        "direction_rationale_counts": direction_rationale_counts,
        "field_mismatch_counts": field_mismatch_counts,
        "field_mismatches": field_mismatches,
        "recomputed_field_counts": recomputed_field_counts,
    }

analysis_by_repo = {
    repo: analyze_repo(repo, manifest_rows)
    for repo, manifest_rows in manifest_rows_by_repo.items()
}

ACTIVE_REPO = "sigma"
joined_rows = analysis_by_repo[ACTIVE_REPO]["joined_rows"]
missing_keys = analysis_by_repo[ACTIVE_REPO]["missing_keys"]
direction_rationale_counts = analysis_by_repo[ACTIVE_REPO]["direction_rationale_counts"]
field_mismatch_counts = analysis_by_repo[ACTIVE_REPO]["field_mismatch_counts"]
field_mismatches = analysis_by_repo[ACTIVE_REPO]["field_mismatches"]
recomputed_field_counts = analysis_by_repo[ACTIVE_REPO]["recomputed_field_counts"]

for repo, analysis in analysis_by_repo.items():
    print(f"{repo}: joined rows = {len(analysis['joined_rows'])}")
    print(f"{repo}: missing keys = {len(analysis['missing_keys'])}")

missing_keys[:10]


sigma: joined rows = 7956
sigma: missing keys = 2
ssc: joined rows = 4412
ssc: missing keys = 0


[('sigma', 'lineage_02536', 1, 2), ('sigma', 'lineage_02536', 7, 8)]

## 1. `match_set_direction` vs `rationale_label`

This section now reports the direction/rationale combinations separately for `sigma` and `ssc`.


In [3]:
for repo, analysis in analysis_by_repo.items():
    print(f"== {repo} ==")
    for (direction, rationale), count in sorted(analysis["direction_rationale_counts"].items()):
        print(f"{direction:8s} | {rationale:24s} | {count}")
    print()


== sigma ==
broader  | coverage_expansion       | 3287
broader  | insufficient_evidence    | 7
broader  | mixed_tradeoff           | 14
mixed    | false_positive_reduction | 1
mixed    | insufficient_evidence    | 43
mixed    | mixed_tradeoff           | 921
narrower | false_positive_reduction | 2400
narrower | insufficient_evidence    | 387
narrower | mixed_tradeoff           | 16
unclear  | insufficient_evidence    | 879
unclear  | mixed_tradeoff           | 1

== ssc ==
broader  | coverage_expansion       | 1586
broader  | insufficient_evidence    | 8
broader  | mixed_tradeoff           | 3
mixed    | insufficient_evidence    | 27
mixed    | mixed_tradeoff           | 488
narrower | false_positive_reduction | 793
narrower | insufficient_evidence    | 321
narrower | mixed_tradeoff           | 14
unclear  | insufficient_evidence    | 1172



## 2. Field Matches Recomputed From Raw PG-IR Ops

This section no longer uses precomputed `pgir_labels` from `test.json`.

Instead, for each manifest row, it recomputes the active PG-IR ops from the original files and checks whether those ops support the updated LLM fields:

- `predicate_modified_present` supported by `PRED_UPDATE`
- `predicate_added` supported by `AND_ADD`, `OR_ADD`, `BRANCH_ADD`
- `predicate_removed` supported by `AND_REMOVE`, `OR_REMOVE`, `BRANCH_REMOVE`

`value_added_present` and `value_removed_present` are intentionally not part of this mismatch check.

Mismatch definition used here:
A mismatch is counted only when the LLM asserts a coarse label and the recomputed PG-IR ops do not provide supporting evidence for that label. Extra PG-IR signals that the LLM did not mark are not counted as mismatches in this section.

If one of these updated LLM fields is missing from a row, the code below treats it as `False` so the notebook can still run against partially migrated manifests.


In [4]:
for repo, analysis in analysis_by_repo.items():
    print(f"== {repo} ==")
    for field, count in analysis["field_mismatch_counts"].items():
        print(f"{field:28s} -> {count}")
    print("total mismatches:", len(analysis["field_mismatches"]))
    print()


== sigma ==
predicate_modified_present   -> 1315
predicate_added              -> 585
predicate_removed            -> 303
total mismatches: 2203

== ssc ==
predicate_modified_present   -> 793
predicate_added              -> 66
predicate_removed            -> 58
total mismatches: 917



## 3. Raw Op Frequency By Repo

This is a quick view of how often each recomputed coarse signal appears in the `sigma` and `ssc` result files.


In [5]:
for repo, analysis in analysis_by_repo.items():
    print(f"== {repo} ==")
    for field, count in sorted(analysis["recomputed_field_counts"].items()):
        print(f"{field:28s} -> {count}")
    print()


== sigma ==
predicate_added              -> 4269
predicate_modified_present   -> 5002
predicate_removed            -> 3840

== ssc ==
predicate_added              -> 2870
predicate_modified_present   -> 2513
predicate_removed            -> 2768



## 4. Validation Summary (Table 9)

This table combines the predicate-change agreement audit from `intention.ipynb`
with the repo-specific field-mismatch ratios recomputed in this notebook.
The downstream direction and rationale analyses use only the changed-pair subset
where both PG-IR and the LLM agree that predicate logic changed.


In [8]:
AGGREGATE_RESULT_PATHS = {
    repo: ROOT / "data_prep" / "llm_data" / repo / "aggregate_results.jsonl"
    for repo in REPO_RESULT_PATHS
}


def load_all_steps(repo: str) -> list[dict]:
    path = ROOT / "data_prep" / "align_data" / f"all_steps_{repo}.jsonl"
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_aggregate_results(repo: str) -> tuple[dict, dict]:
    by_key = {}
    by_lineage = {}
    with AGGREGATE_RESULT_PATHS[repo].open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            key = (rec["lineage_id"], rec["version_a"], rec["version_b"])
            by_key[key] = rec
            by_lineage.setdefault(rec["lineage_id"], {})[(rec["version_a"], rec["version_b"])] = rec
    return by_key, by_lineage


def infer_llm_predicate_change(step: dict, aggregate_by_key: dict, aggregate_by_lineage: dict):
    """Return (llm_predicate_changed, source) for one temporal step.

    Exact matches use the aggregate LLM row directly.  SSC also contains a few
    non-adjacent jump steps in all_steps where empty PG-IR intermediate versions
    were collapsed out (for example 10->12).  For those, infer a step-level LLM
    label from the full adjacent path if every constituent adjacent pair is
    available in aggregate_results.
    """
    key = (step["lineage_id"], step["version_a"], step["version_b"])
    rec = aggregate_by_key.get(key)
    if rec is not None:
        if rec.get("llm_error") is not None:
            return None, "llm_error"
        llm_pred = (rec.get("llm_result") or {}).get("predicate_logic_changed")
        return llm_pred, "exact"

    if step["version_b"] <= step["version_a"] + 1:
        return None, "missing_exact"

    preds = []
    lineage_rows = aggregate_by_lineage.get(step["lineage_id"], {})
    for va in range(step["version_a"], step["version_b"]):
        sub = lineage_rows.get((va, va + 1))
        if sub is None:
            return None, "missing_adjacent"
        if sub.get("llm_error") is not None:
            return None, "adjacent_llm_error"
        sub_pred = (sub.get("llm_result") or {}).get("predicate_logic_changed")
        if sub_pred is None:
            return None, "missing_adjacent_label"
        preds.append(bool(sub_pred))

    return any(preds), "inferred_jump"


def compute_pred_change_agreement(repo: str) -> dict:
    steps = load_all_steps(repo)
    aggregate_by_key, aggregate_by_lineage = load_aggregate_results(repo)

    agreed = 0
    total = 0
    inferred_jump_pairs = 0
    excluded_reasons = Counter()

    for step in steps:
        llm_pred, source = infer_llm_predicate_change(step, aggregate_by_key, aggregate_by_lineage)
        if llm_pred is None:
            excluded_reasons[source] += 1
            continue

        if source == "inferred_jump":
            inferred_jump_pairs += 1

        total += 1
        pgir_pred = float(step.get("d_step", 0.0)) > 0.0
        if pgir_pred == bool(llm_pred):
            agreed += 1

    return {
        "agreed": agreed,
        "total": total,
        "all_steps": len(steps),
        "excluded": sum(excluded_reasons.values()),
        "excluded_reasons": dict(excluded_reasons),
        "inferred_jump_pairs": inferred_jump_pairs,
    }


pred_change_agreement_by_repo = {
    repo: compute_pred_change_agreement(repo)
    for repo in ("ssc", "sigma")
}

validation_rows = []
validation_rows.append({
    "metric": "Pred-change agreement (PG-IR d_pred>0 vs LLM predicate_logic_changed)",
    "ssc": (
        f"{pred_change_agreement_by_repo['ssc']['agreed']}/"
        f"{pred_change_agreement_by_repo['ssc']['total']} "
        f"({pred_change_agreement_by_repo['ssc']['agreed'] / pred_change_agreement_by_repo['ssc']['total'] * 100:.1f}%)"
    ),
    "sigma": (
        f"{pred_change_agreement_by_repo['sigma']['agreed']}/"
        f"{pred_change_agreement_by_repo['sigma']['total']} "
        f"({pred_change_agreement_by_repo['sigma']['agreed'] / pred_change_agreement_by_repo['sigma']['total'] * 100:.1f}%)"
    ),
    "notes": (
        "Computed from aggregate_results against all temporal step rows. "
        "SSC includes 10 inferred jump-step pairs from collapsed empty PG-IR versions; "
        "Sigma excludes 3 oversized-prompt LLM error rows. "
        "Pairs retained for downstream direction/rationale analysis are the agreed-changed subset."
    ),
})

for field in llm_field_support:
    row = {
        "metric": f"{field} mismatches / structurally supported positives",
        "notes": "Mismatch counted only when the LLM asserts the field but recomputed PG-IR ops do not support it.",
    }
    for repo, analysis in analysis_by_repo.items():
        mismatch_count = analysis["field_mismatch_counts"].get(field, 0)
        support_total = analysis["recomputed_field_counts"].get(field, 0)
        pct = (mismatch_count / support_total * 100) if support_total else float("nan")
        row[repo] = f"{mismatch_count}/{support_total} ({pct:.1f}%)" if support_total else "N/A"
    validation_rows.append(row)

display_table(validation_rows, descending=False)


,metric,ssc,sigma,notes
0,Pred-change agreement (PG-IR d_pred>0 vs LLM p...,105854/107086 (98.8%),39181/39477 (99.3%),Computed from aggregate_results against all te...
1,predicate_modified_present mismatches / struct...,793/2513 (31.6%),1315/5002 (26.3%),Mismatch counted only when the LLM asserts the...
2,predicate_added mismatches / structurally supp...,66/2870 (2.3%),585/4269 (13.7%),Mismatch counted only when the LLM asserts the...
3,predicate_removed mismatches / structurally su...,58/2768 (2.1%),303/3840 (7.9%),Mismatch counted only when the LLM asserts the...


[{'metric': 'Pred-change agreement (PG-IR d_pred>0 vs LLM predicate_logic_changed)',
  'ssc': '105854/107086 (98.8%)',
  'sigma': '39181/39477 (99.3%)',
  'notes': 'Computed from aggregate_results against all temporal step rows. SSC includes 10 inferred jump-step pairs from collapsed empty PG-IR versions; Sigma excludes 3 oversized-prompt LLM error rows. Pairs retained for downstream direction/rationale analysis are the agreed-changed subset.'},
 {'metric': 'predicate_modified_present mismatches / structurally supported positives',
  'notes': 'Mismatch counted only when the LLM asserts the field but recomputed PG-IR ops do not support it.',
  'sigma': '1315/5002 (26.3%)',
  'ssc': '793/2513 (31.6%)'},
 {'metric': 'predicate_added mismatches / structurally supported positives',
  'notes': 'Mismatch counted only when the LLM asserts the field but recomputed PG-IR ops do not support it.',
  'sigma': '585/4269 (13.7%)',
  'ssc': '66/2870 (2.3%)'},
 {'metric': 'predicate_removed mismatches 

# Aggregate Intent Distribution

## SSC and Sigma Lineage-Level Patterns

This section looks at both:
- `data_prep/llm_data/ssc/pair_changed_results.jsonl`
- `data_prep/llm_data/sigma/pair_changed_results.jsonl`

For each dataset, we group results by `lineage_id` so we can ask how each rule evolves across versions.

The goal is to find patterns like:
- lineages that are consistently about `coverage_expansion` or consistently about `false_positive_reduction` with `insufficient_evidence` mixed in
- lineages that flip between `coverage_expansion` and `false_positive_reduction`
- lineages where `mixed_tradeoff` is the dominant explanation
- lineages whose `match_set_direction` is one-sided versus those that contain both `broader` and `narrower`


In [10]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from pprint import pprint

try:
    import pandas as pd  # Optional, only for prettier notebook tables.
except ModuleNotFoundError:
    pd = None


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data_prep" / "llm_data" / "ssc").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data_prep/llm_data/ssc")


def display_table(rows, sort_by=None, descending=True, limit=None):
    rows = list(rows)
    if sort_by is not None:
        rows = sorted(rows, key=lambda row: row[sort_by], reverse=descending)
    if limit is not None:
        rows = rows[:limit]
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        pprint(rows)
    return rows

def load_results(path: Path):
    rows = []
    with path.open() as f:
        for line in f:
            obj = json.loads(line)
            row = {key: obj.get(key) for key in KEY_FIELDS + ["model", "llm_error"]}
            llm_result = obj.get("llm_result") or {}
            for field, value in llm_result.items():
                if field == "summary":
                    continue
                row[field] = value
            rows.append(row)
    return rows

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data_prep" / "llm_data" / "ssc"


In [12]:
DATASET_CONFIG = {
    "ssc": {
        "title": "SSC",
        "pair_path": REPO_ROOT / "data_prep" / "llm_data" / "ssc" / "pair_changed_results.jsonl",
        "rule_versions_path": REPO_ROOT / "data_prep" / "build_data" / "rule_versions_ssc.jsonl",
    },
    "sigma": {
        "title": "Sigma",
        "pair_path": REPO_ROOT / "data_prep" / "llm_data" / "sigma" / "pair_changed_results.jsonl",
        "rule_versions_path": REPO_ROOT / "data_prep" / "build_data" / "rule_versions_sigma.jsonl",
    },
}

KEY_FIELDS = [
    "lineage_id",
    "rule_canonical",
    "version_a",
    "version_b",
    "commit_a",
    "commit_b",
] 
LINEAGE_DATASETS = {}
for dataset_name, config in DATASET_CONFIG.items():
    pair_rows = load_results(config["pair_path"])

    by_lineage = {}
    for row in pair_rows:
        lineage_id = row["lineage_id"]
        by_lineage.setdefault(lineage_id, []).append(row)

    for lineage_rows in by_lineage.values():
        lineage_rows.sort(key=lambda row: (row["version_a"], row["version_b"]))

    LINEAGE_DATASETS[dataset_name] = {
        "pair_rows": pair_rows,
        "by_lineage": by_lineage,
        "rule_versions_path": config["rule_versions_path"],
        "title": config["title"],
    }

    print(f"{config['title']} rows: {len(pair_rows):,}")
    print(f"{config['title']} lineages: {len(by_lineage):,}")
    print()


SSC rows: 4,412
SSC lineages: 1,451

Sigma rows: 7,958
Sigma lineages: 2,338



In [13]:
def _direction_change_count(labels):
    return sum(left != right for left, right in zip(labels, labels[1:]))


ALL_LINEAGE_BUCKETS = [
    "IE-only",
    "Singleton (1 non-IE revision)",
    "Multi-revision (>=2 non-IE)",
]

MULTI_REVISION_BUCKETS = [
    "Coupled (>=50% MT)",
    "CE-only",
    "FPR-only",
    "Alternating - Oscillating (k >= 2)",
    "Alternating - Phased (k = 1)",
]

SINGLETON_BUCKETS = [
    "CE singleton",
    "FPR singleton",
    "MT singleton",
]


def build_lineage_pattern_rows(by_lineage):
    pattern_rows = []
    for lineage_id, lineage_rows in by_lineage.items():
        labels = [row.get("rationale_label") for row in lineage_rows]
        directions = [row.get("match_set_direction") for row in lineage_rows]
        label_counts = Counter(labels)
        direction_counts = Counter(directions)
        label_set = set(labels)
        direction_set = set(directions)

        non_ie_labels = [
            label for label in labels
            if label != "insufficient_evidence"
        ]
        directional_labels = [
            label for label in non_ie_labels
            if label in {"coverage_expansion", "false_positive_reduction"}
        ]
        directional_set = set(directional_labels)

        n_non_ie = len(non_ie_labels)
        n_directional = len(directional_labels)
        mixed_tradeoff_count = label_counts["mixed_tradeoff"]
        ce_fpr_count = label_counts["coverage_expansion"] + label_counts["false_positive_reduction"]
        mixed_tradeoff_share = (
            label_counts["mixed_tradeoff"] / len(lineage_rows)
            if lineage_rows else 0.0
        )
        mixed_tradeoff_share_non_ie = (
            mixed_tradeoff_count / n_non_ie
            if n_non_ie else 0.0
        )
        direction_change_count = _direction_change_count(directional_labels)

        only_insufficient_evidence = label_set == {"insufficient_evidence"}
        is_singleton_non_ie = n_non_ie == 1
        is_multi_revision = n_non_ie >= 2
        has_both_expansion_and_fpr = directional_set == {"coverage_expansion", "false_positive_reduction"}
        predominantly_mixed_tradeoff = is_multi_revision and mixed_tradeoff_share_non_ie >= 0.5
        only_coverage_or_unclear = directional_set == {"coverage_expansion"}
        only_false_positive_reduction_or_unclear = directional_set == {"false_positive_reduction"}
        is_oscillating = has_both_expansion_and_fpr and direction_change_count >= 2
        is_phased = has_both_expansion_and_fpr and direction_change_count == 1

        if only_insufficient_evidence:
            cohort_a_bucket = "IE-only"
            singleton_bucket = None
            multi_revision_taxonomy = None
        elif is_singleton_non_ie:
            cohort_a_bucket = "Singleton (1 non-IE revision)"
            if non_ie_labels[0] == "coverage_expansion":
                singleton_bucket = "CE singleton"
            elif non_ie_labels[0] == "false_positive_reduction":
                singleton_bucket = "FPR singleton"
            else:
                singleton_bucket = "MT singleton"
            multi_revision_taxonomy = None
        else:
            cohort_a_bucket = "Multi-revision (>=2 non-IE)"
            singleton_bucket = None
            if predominantly_mixed_tradeoff:
                multi_revision_taxonomy = "Coupled (>=50% MT)"
            elif only_coverage_or_unclear:
                multi_revision_taxonomy = "CE-only"
            elif only_false_positive_reduction_or_unclear:
                multi_revision_taxonomy = "FPR-only"
            elif is_oscillating:
                multi_revision_taxonomy = "Alternating - Oscillating (k >= 2)"
            elif is_phased:
                multi_revision_taxonomy = "Alternating - Phased (k = 1)"
            else:
                raise ValueError(
                    f"Unclassified multi-revision lineage {lineage_id}: labels={labels}"
                )

        pattern_rows.append({
            "lineage_id": lineage_id,
            "rule_canonical": lineage_rows[0]["rule_canonical"],
            "n_pairs": len(lineage_rows),
            "n_non_ie": n_non_ie,
            "label_sequence": " -> ".join(labels),
            "direction_sequence": " -> ".join(directions),
            "label_counts": dict(label_counts),
            "direction_counts": dict(direction_counts),
            "non_ie_label_sequence": " -> ".join(non_ie_labels),
            "filtered_expansion_fpr_sequence": " -> ".join(directional_labels),
            "directional_sequence": " -> ".join(directional_labels),
            "only_coverage_or_unclear": only_coverage_or_unclear,
            "only_false_positive_reduction_or_unclear": only_false_positive_reduction_or_unclear,
            "only_insufficient_evidence": only_insufficient_evidence,
            "has_both_expansion_and_fpr": has_both_expansion_and_fpr,
            "predominantly_mixed_tradeoff": predominantly_mixed_tradeoff,
            "mixed_tradeoff_share": round(mixed_tradeoff_share, 3),
            "mixed_tradeoff_share_non_ie": round(mixed_tradeoff_share_non_ie, 3),
            "mixed_tradeoff_count": mixed_tradeoff_count,
            "ce_fpr_count": ce_fpr_count,
            "only_broader_or_unclear": direction_set <= {"broader", "unclear"} and "broader" in direction_set,
            "only_narrower_or_unclear": direction_set <= {"narrower", "unclear"} and "narrower" in direction_set,
            "has_both_broader_and_narrower": (
                "broader" in direction_set and "narrower" in direction_set
            ),
            "alternating_expansion_vs_fpr": has_both_expansion_and_fpr,
            "direction_change_count": direction_change_count,
            "n_flips": direction_change_count,
            "n_rel_labels": n_directional,
            "oscillation_prominence": (direction_change_count, n_directional, len(lineage_rows)),
            "mixed_prominence": (mixed_tradeoff_count, mixed_tradeoff_share, len(lineage_rows)),
            "is_oscillating": is_oscillating,
            "is_phased": is_phased,
            "cohort_a_bucket": cohort_a_bucket,
            "singleton_bucket": singleton_bucket,
            "multi_revision_taxonomy": multi_revision_taxonomy,
        })
    return pattern_rows


def _build_summary_rows(rows, patterns, key, description_map, total, cohort):
    summary = []
    for pattern in patterns:
        lineage_count = sum(row.get(key) == pattern for row in rows)
        summary.append({
            "cohort": cohort,
            "pattern": pattern,
            "description": description_map[pattern],
            "lineage_count": lineage_count,
            "pct_of_lineages": round(lineage_count / total * 100, 2) if total else float("nan"),
        })
    return summary


def build_lineage_pattern_summary(pattern_rows):
    all_descriptions = {
        "IE-only": "Every revision is insufficient-evidence",
        "Singleton (1 non-IE revision)": "Exactly one non-IE revision; trajectory is undefined",
        "Multi-revision (>=2 non-IE)": "At least two non-IE revisions; lineage enters the trajectory taxonomy",
    }
    multi_descriptions = {
        "Coupled (>=50% MT)": "Within-revision tradeoff dominates: at least half of non-IE revisions are MT",
        "CE-only": "After dropping IE and MT, all directional revisions are CE",
        "FPR-only": "After dropping IE and MT, all directional revisions are FPR",
        "Alternating - Oscillating (k >= 2)": "Directional revisions include both CE and FPR and flip at least twice",
        "Alternating - Phased (k = 1)": "Directional revisions include both CE and FPR with one regime change and no return",
    }
    singleton_descriptions = {
        "CE singleton": "The sole non-IE revision is CE",
        "FPR singleton": "The sole non-IE revision is FPR",
        "MT singleton": "The sole non-IE revision is MT",
    }

    multi_revision_rows = [
        row for row in pattern_rows
        if row["cohort_a_bucket"] == "Multi-revision (>=2 non-IE)"
    ]
    singleton_rows = [
        row for row in pattern_rows
        if row["cohort_a_bucket"] == "Singleton (1 non-IE revision)"
    ]

    return {
        "all_lineages": _build_summary_rows(
            pattern_rows,
            ALL_LINEAGE_BUCKETS,
            "cohort_a_bucket",
            all_descriptions,
            len(pattern_rows),
            "All lineages",
        ),
        "multi_revision": _build_summary_rows(
            multi_revision_rows,
            MULTI_REVISION_BUCKETS,
            "multi_revision_taxonomy",
            multi_descriptions,
            len(multi_revision_rows),
            "Multi-revision lineages only",
        ),
        "singleton_breakdown": _build_summary_rows(
            singleton_rows,
            SINGLETON_BUCKETS,
            "singleton_bucket",
            singleton_descriptions,
            len(singleton_rows),
            "Singleton breakdown",
        ),
    }


for dataset_name, dataset in LINEAGE_DATASETS.items():
    pattern_rows = build_lineage_pattern_rows(dataset["by_lineage"])
    dataset["pattern_rows"] = pattern_rows
    dataset["pattern_summary"] = build_lineage_pattern_summary(pattern_rows)
    print(f"{dataset['title']} lineage pattern rows: {len(pattern_rows):,}")


SSC lineage pattern rows: 1,451
Sigma lineage pattern rows: 2,338


In [15]:
lineage_pattern_summary_rows = []
for dataset_name, dataset in LINEAGE_DATASETS.items():
    for cohort_name in ["all_lineages", "multi_revision", "singleton_breakdown"]:
        for row in dataset["pattern_summary"][cohort_name]:
            lineage_pattern_summary_rows.append({
                "dataset": dataset["title"],
                **row,
            })

display_table(lineage_pattern_summary_rows, descending=False)


,dataset,cohort,pattern,description,lineage_count,pct_of_lineages
0,SSC,All lineages,IE-only,Every revision is insufficient-evidence,212,14.61
1,SSC,All lineages,Singleton (1 non-IE revision),Exactly one non-IE revision; trajectory is und...,563,38.80
2,SSC,All lineages,Multi-revision (>=2 non-IE),At least two non-IE revisions; lineage enters ...,676,46.59
3,SSC,Multi-revision lineages only,Coupled (>=50% MT),Within-revision tradeoff dominates: at least h...,119,17.60
4,SSC,Multi-revision lineages only,CE-only,"After dropping IE and MT, all directional revi...",156,23.08
5,SSC,Multi-revision lineages only,FPR-only,"After dropping IE and MT, all directional revi...",21,3.11
6,SSC,Multi-revision lineages only,Alternating - Oscillating (k >= 2),Directional revisions include both CE and FPR ...,159,23.52
7,SSC,Multi-revision lineages only,Alternating - Phased (k = 1),Directional revisions include both CE and FPR ...,221,32.69
8,SSC,Singleton breakdown,CE singleton,The sole non-IE revision is CE,339,60.21
9,SSC,Singleton breakdown,FPR singleton,The sole non-IE revision is FPR,151,26.82


[{'dataset': 'SSC',
  'cohort': 'All lineages',
  'pattern': 'IE-only',
  'description': 'Every revision is insufficient-evidence',
  'lineage_count': 212,
  'pct_of_lineages': 14.61},
 {'dataset': 'SSC',
  'cohort': 'All lineages',
  'pattern': 'Singleton (1 non-IE revision)',
  'description': 'Exactly one non-IE revision; trajectory is undefined',
  'lineage_count': 563,
  'pct_of_lineages': 38.8},
 {'dataset': 'SSC',
  'cohort': 'All lineages',
  'pattern': 'Multi-revision (>=2 non-IE)',
  'description': 'At least two non-IE revisions; lineage enters the trajectory taxonomy',
  'lineage_count': 676,
  'pct_of_lineages': 46.59},
 {'dataset': 'SSC',
  'cohort': 'Multi-revision lineages only',
  'pattern': 'Coupled (>=50% MT)',
  'description': 'Within-revision tradeoff dominates: at least half of non-IE revisions are MT',
  'lineage_count': 119,
  'pct_of_lineages': 17.6},
 {'dataset': 'SSC',
  'cohort': 'Multi-revision lineages only',
  'pattern': 'CE-only',
  'description': 'After d

## Lineage Pattern Taxonomy (Table 7)

We distinguish within-revision coupling from across-revision alternation. The first
cohort reports all lineages as `IE-only`, `Singleton`, or `Multi-revision`; the
second restricts to lineages with `>=2` non-IE revisions and then applies a
priority-ordered, mutually exclusive taxonomy: `Coupled`, `CE-only`, `FPR-only`,
and `Alternating`, split into `Oscillating` (`k >= 2`) versus `Phased` (`k = 1`).


In [16]:
from statistics import median


for dataset in LINEAGE_DATASETS.values():
    if not dataset.get("pattern_rows") or "cohort_a_bucket" not in dataset["pattern_rows"][0]:
        dataset["pattern_rows"] = build_lineage_pattern_rows(dataset["by_lineage"])
        dataset["pattern_summary"] = build_lineage_pattern_summary(dataset["pattern_rows"])


dataset_titles = [dataset["title"] for dataset in LINEAGE_DATASETS.values()]


def _lookup_summary_row(dataset, cohort_name, pattern):
    for row in dataset["pattern_summary"][cohort_name]:
        if row["pattern"] == pattern:
            return row
    raise KeyError((dataset["title"], cohort_name, pattern))


taxonomy_table_rows = []
for cohort_label, row_label, cohort_name, pattern in [
    ("All lineages", "Total lineages", None, None),
    ("All lineages", "IE-only", "all_lineages", "IE-only"),
    ("All lineages", "Singleton (1 non-IE revision)", "all_lineages", "Singleton (1 non-IE revision)"),
    ("All lineages", "Multi-revision (>=2 non-IE)", "all_lineages", "Multi-revision (>=2 non-IE)"),
    ("Multi-revision lineages only", "Total", None, None),
    ("Multi-revision lineages only", "Coupled (>=50% MT)", "multi_revision", "Coupled (>=50% MT)"),
    ("Multi-revision lineages only", "CE-only", "multi_revision", "CE-only"),
    ("Multi-revision lineages only", "FPR-only", "multi_revision", "FPR-only"),
    ("Multi-revision lineages only", "Alternating - Oscillating (k >= 2)", "multi_revision", "Alternating - Oscillating (k >= 2)"),
    ("Multi-revision lineages only", "Alternating - Phased (k = 1)", "multi_revision", "Alternating - Phased (k = 1)"),
]:
    table_row = {
        "cohort": cohort_label,
        "row": row_label,
    }

    for dataset in LINEAGE_DATASETS.values():
        title = dataset["title"]
        total_lineages = len(dataset["pattern_rows"])
        multi_revision_total = sum(
            row["cohort_a_bucket"] == "Multi-revision (>=2 non-IE)"
            for row in dataset["pattern_rows"]
        )

        if row_label == "Total lineages":
            count = total_lineages
            pct = 100.0
        elif row_label == "Total":
            count = multi_revision_total
            pct = 100.0
        else:
            summary_row = _lookup_summary_row(dataset, cohort_name, pattern)
            count = summary_row["lineage_count"]
            pct = summary_row["pct_of_lineages"]

        table_row[f"{title} count"] = count
        table_row[f"{title} pct"] = pct
    taxonomy_table_rows.append(table_row)

display_table(taxonomy_table_rows, descending=False)

print()
for dataset_name, dataset in LINEAGE_DATASETS.items():
    title = dataset["title"]
    singleton_rows = [
        row for row in dataset["pattern_rows"]
        if row["cohort_a_bucket"] == "Singleton (1 non-IE revision)"
    ]
    singleton_total = len(singleton_rows)
    singleton_counts = Counter(row["singleton_bucket"] for row in singleton_rows)
    coupled_rows = [
        row for row in dataset["pattern_rows"]
        if row["multi_revision_taxonomy"] == "Coupled (>=50% MT)"
    ]

    print(f"== {title} ==")
    print(
        "Singleton breakdown: "
        f"CE={singleton_counts['CE singleton']}, "
        f"FPR={singleton_counts['FPR singleton']}, "
        f"MT={singleton_counts['MT singleton']} "
        f"(total={singleton_total})"
    )
    if coupled_rows:
        mt_shares = [row["mixed_tradeoff_share_non_ie"] for row in coupled_rows]
        print(
            "Coupled MT-share among non-IE revisions: "
            f"min={min(mt_shares):.2f}, median={median(mt_shares):.2f}, max={max(mt_shares):.2f}"
        )
    print()


,cohort,row,SSC count,SSC pct,Sigma count,Sigma pct
0,All lineages,Total lineages,1451,100.00,2338,100.00
1,All lineages,IE-only,212,14.61,248,10.61
2,All lineages,Singleton (1 non-IE revision),563,38.80,845,36.14
3,All lineages,Multi-revision (>=2 non-IE),676,46.59,1245,53.25
4,Multi-revision lineages only,Total,676,100.00,1245,100.00
5,Multi-revision lineages only,Coupled (>=50% MT),119,17.60,149,11.97
6,Multi-revision lineages only,CE-only,156,23.08,312,25.06
7,Multi-revision lineages only,FPR-only,21,3.11,86,6.91
8,Multi-revision lineages only,Alternating - Oscillating (k >= 2),159,23.52,344,27.63
9,Multi-revision lineages only,Alternating - Phased (k = 1),221,32.69,354,28.43



== SSC ==
Singleton breakdown: CE=339, FPR=151, MT=73 (total=563)
Coupled MT-share among non-IE revisions: min=0.50, median=0.50, max=1.00

== Sigma ==
Singleton breakdown: CE=519, FPR=215, MT=111 (total=845)
Coupled MT-share among non-IE revisions: min=0.50, median=0.50, max=1.00



## Time Aspect of Trajectory (Table 8)

In [19]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt


def parse_commit_dt(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00")) if value else None


def percentile(values, p):
    values = sorted(values)
    if not values:
        return None
    if len(values) == 1:
        return values[0]
    idx = (len(values) - 1) * p
    lower = int(idx)
    upper = min(lower + 1, len(values) - 1)
    frac = idx - lower
    return values[lower] * (1 - frac) + values[upper] * frac


def summarize_days(values):
    values = sorted(value for value in values if value is not None)
    if not values:
        return None
    return {
        "n": len(values),
        "median_days": median(values),
        "p25_days": percentile(values, 0.25),
        "p75_days": percentile(values, 0.75),
        "mean_days": sum(values) / len(values),
        "max_days": max(values),
    }


def summarize_binary(flags):
    flags = list(flags)
    total = len(flags)
    true_count = sum(bool(flag) for flag in flags)
    false_count = total - true_count
    return {
        "n": total,
        "true_count": true_count,
        "false_count": false_count,
        "true_pct": round(true_count / total * 100, 2) if total else float("nan"),
    }


def round_table_rows(rows, decimals=2):
    rounded_rows = []
    for row in rows:
        rounded = {}
        for key, value in row.items():
            if isinstance(value, float):
                rounded[key] = round(value, decimals)
            else:
                rounded[key] = value
        rounded_rows.append(rounded)
    return rounded_rows


def markdown_table(rows):
    rows = list(rows)
    if not rows:
        return "(no rows)\n"
    columns = list(rows[0].keys())
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        values = []
        for column in columns:
            value = row.get(column, "")
            if value is None:
                values.append("")
            else:
                values.append(str(value).replace("\n", " "))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body]) + "\n"


def attach_temporal_rows(dataset_name, dataset):
    rule_dates = {}
    with dataset["rule_versions_path"].open(encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            rule_dates[(row["lineage_id"], int(row["version_index"]))] = parse_commit_dt(row.get("commit_date"))

    temporal_pair_rows = []
    temporal_by_lineage = defaultdict(list)
    for row in dataset["pair_rows"]:
        key_a = (row["lineage_id"], int(row["version_a"]))
        key_b = (row["lineage_id"], int(row["version_b"]))
        dt_a = rule_dates.get(key_a)
        dt_b = rule_dates.get(key_b)
        if dt_a is None or dt_b is None:
            continue

        enriched = dict(row)
        enriched["commit_date_a"] = dt_a
        enriched["commit_date_b"] = dt_b
        enriched["gap_days"] = (dt_b - dt_a).total_seconds() / 86400.0
        temporal_pair_rows.append(enriched)
        temporal_by_lineage[row["lineage_id"]].append(enriched)

    for lineage_rows in temporal_by_lineage.values():
        lineage_rows.sort(key=lambda row: (int(row["version_a"]), int(row["version_b"])))

    dataset["temporal_pair_rows"] = temporal_pair_rows
    dataset["temporal_by_lineage"] = dict(temporal_by_lineage)
    dataset["pattern_by_lineage"] = {row["lineage_id"]: row for row in dataset["pattern_rows"]}


def lineage_span_days(lineage_rows):
    return (lineage_rows[-1]["commit_date_b"] - lineage_rows[0]["commit_date_a"]).total_seconds() / 86400.0


def lineage_revision_pace_days(lineage_rows):
    n_revisions = len(lineage_rows) + 1
    if n_revisions <= 1:
        return None
    return lineage_span_days(lineage_rows) / (n_revisions - 1)


def directional_rows_only(lineage_rows):
    return [
        row for row in lineage_rows
        if row.get("rationale_label") in {"coverage_expansion", "false_positive_reduction"}
    ]


def mature_lineage_ids(dataset, min_days=730):
    ids = []
    for lineage_id, lineage_rows in dataset["temporal_by_lineage"].items():
        if lineage_span_days(lineage_rows) >= min_days:
            ids.append(lineage_id)
    return ids


for dataset_name, dataset in LINEAGE_DATASETS.items():
    attach_temporal_rows(dataset_name, dataset)


pair_gap_rows = []
for dataset_name, dataset in LINEAGE_DATASETS.items():
    for label in ["coverage_expansion", "false_positive_reduction", "mixed_tradeoff", "insufficient_evidence"]:
        summary = summarize_days(
            row["gap_days"]
            for row in dataset["temporal_pair_rows"]
            if row.get("rationale_label") == label
        )
        pair_gap_rows.append({
            "dataset": dataset["title"],
            "label": label,
            **summary,
        })

print("\nTable T1. Pair-Level Revision Gap by Rationale Label")
display_table(round_table_rows(pair_gap_rows), descending=False)


transition_order = [
    "coverage_expansion -> coverage_expansion",
    "false_positive_reduction -> false_positive_reduction",
    "coverage_expansion -> false_positive_reduction",
    "false_positive_reduction -> coverage_expansion",
]
transition_short = {
    "coverage_expansion -> coverage_expansion": "CE -> CE",
    "false_positive_reduction -> false_positive_reduction": "FPR -> FPR",
    "coverage_expansion -> false_positive_reduction": "CE -> FPR",
    "false_positive_reduction -> coverage_expansion": "FPR -> CE",
}

transition_gap_rows = []
lineage_transition_rows = []
lineage_pace_rows = []
for dataset_name, dataset in LINEAGE_DATASETS.items():
    transition_gap_by_type = defaultdict(list)
    lineage_transition_gap_by_type = defaultdict(list)
    pace_by_taxonomy = defaultdict(list)

    for lineage_id, lineage_rows in dataset["temporal_by_lineage"].items():
        pattern_row = dataset["pattern_by_lineage"][lineage_id]
        taxonomy = pattern_row["multi_revision_taxonomy"] or pattern_row["cohort_a_bucket"]
        pace_by_taxonomy[taxonomy].append(lineage_revision_pace_days(lineage_rows))

        directional_rows = directional_rows_only(lineage_rows)
        for left, right in zip(directional_rows, directional_rows[1:]):
            transition_key = f"{left['rationale_label']} -> {right['rationale_label']}"
            transition_gap = (right["commit_date_b"] - left["commit_date_b"]).total_seconds() / 86400.0
            transition_gap_by_type[transition_key].append(transition_gap)
            if taxonomy in {"CE-only", "FPR-only", "Alternating - Oscillating (k >= 2)"}:
                lineage_transition_gap_by_type[(taxonomy, transition_key)].append(transition_gap)

    for transition in transition_order:
        summary = summarize_days(transition_gap_by_type[transition])
        transition_gap_rows.append({
            "dataset": dataset["title"],
            "transition": transition,
            "transition_short": transition_short[transition],
            **summary,
        })

    for taxonomy in ["CE-only", "FPR-only", "Alternating - Oscillating (k >= 2)"]:
        for transition in transition_order:
            values = lineage_transition_gap_by_type[(taxonomy, transition)]
            if not values:
                continue
            summary = summarize_days(values)
            lineage_transition_rows.append({
                "dataset": dataset["title"],
                "taxonomy": taxonomy,
                "transition": transition,
                "transition_short": transition_short[transition],
                **summary,
            })

    for taxonomy in [
        "IE-only",
        "Singleton (1 non-IE revision)",
        "Coupled (>=50% MT)",
        "CE-only",
        "FPR-only",
        "Alternating - Oscillating (k >= 2)",
        "Alternating - Phased (k = 1)",
    ]:
        summary = summarize_days(pace_by_taxonomy[taxonomy])
        lineage_pace_rows.append({
            "dataset": dataset["title"],
            "taxonomy": taxonomy,
            **summary,
        })



reversal_rows = []
stabilization_rows = []
slow_flip_examples = []
for dataset_name, dataset in LINEAGE_DATASETS.items():
    first_flip_by_taxonomy = defaultdict(list)
    terminal_stabilization_flags = []

    for lineage_id, lineage_rows in dataset["temporal_by_lineage"].items():
        pattern_row = dataset["pattern_by_lineage"][lineage_id]
        taxonomy = pattern_row.get("multi_revision_taxonomy")
        if taxonomy not in {"Alternating - Oscillating (k >= 2)", "Alternating - Phased (k = 1)"}:
            continue

        directional_rows = directional_rows_only(lineage_rows)
        if len(directional_rows) < 2:
            continue

        first_date = directional_rows[0]["commit_date_a"]
        first_flip_days = None
        for left, right in zip(directional_rows, directional_rows[1:]):
            if left["rationale_label"] != right["rationale_label"]:
                if first_flip_days is None:
                    first_flip_days = (right["commit_date_b"] - first_date).total_seconds() / 86400.0

        if first_flip_days is not None:
            first_flip_by_taxonomy[taxonomy].append(first_flip_days)
            if taxonomy == "Alternating - Oscillating (k >= 2)":
                terminal_stabilization = directional_rows[-2]["rationale_label"] == directional_rows[-1]["rationale_label"]
                terminal_stabilization_flags.append(terminal_stabilization)
                slow_flip_examples.append({
                    "dataset": dataset["title"],
                    "lineage_id": lineage_id,
                    "rule_canonical": pattern_row["rule_canonical"],
                    "n_pairs": pattern_row["n_pairs"],
                    "n_flips": pattern_row["n_flips"],
                    "terminal_stabilization": terminal_stabilization,
                    "first_flip_days": first_flip_days,
                    "lineage_pace_days": lineage_revision_pace_days(lineage_rows),
                })

    for taxonomy in ["Alternating - Oscillating (k >= 2)", "Alternating - Phased (k = 1)"]:
        first_flip_summary = summarize_days(first_flip_by_taxonomy[taxonomy])
        reversal_rows.append({
            "dataset": dataset["title"],
            "taxonomy": taxonomy,
            "first_flip_n": first_flip_summary["n"],
            "first_flip_median_days": first_flip_summary["median_days"],
            "first_flip_p25_days": first_flip_summary["p25_days"],
            "first_flip_p75_days": first_flip_summary["p75_days"],
        })

    stabilization_summary = summarize_binary(terminal_stabilization_flags)
    stabilization_rows.append({
        "dataset": dataset["title"],
        "oscillating_lineages": stabilization_summary["n"],
        "terminal_stabilization_count": stabilization_summary["true_count"],
        "terminal_stabilization_pct": stabilization_summary["true_pct"],
        "still_flipping_count": stabilization_summary["false_count"],
        "still_flipping_pct": round(100 - stabilization_summary["true_pct"], 2) if stabilization_summary["n"] else float("nan"),
    })



def lookup_row(rows, **filters):
    for row in rows:
        if all(row.get(key) == value for key, value in filters.items()):
            return row
    raise KeyError(filters)


table8_by_dataset = {}
for dataset_title in ["Sigma", "SSC"]:
    ce_ce = lookup_row(
        transition_gap_rows,
        dataset=dataset_title,
        transition="coverage_expansion -> coverage_expansion",
    )
    fpr_fpr = lookup_row(
        transition_gap_rows,
        dataset=dataset_title,
        transition="false_positive_reduction -> false_positive_reduction",
    )
    ce_fpr = lookup_row(
        transition_gap_rows,
        dataset=dataset_title,
        transition="coverage_expansion -> false_positive_reduction",
    )
    fpr_ce = lookup_row(
        transition_gap_rows,
        dataset=dataset_title,
        transition="false_positive_reduction -> coverage_expansion",
    )
    oscillating = lookup_row(
        reversal_rows,
        dataset=dataset_title,
        taxonomy="Alternating - Oscillating (k >= 2)",
    )
    stabilization = lookup_row(stabilization_rows, dataset=dataset_title)

    table8_by_dataset[dataset_title] = {
        "same_direction_gap": f"{int(round(ce_ce['median_days']))} ; {int(round(fpr_fpr['median_days']))}",
        "cross_direction_gap": f"{int(round(ce_fpr['median_days']))} ; {int(round(fpr_ce['median_days']))}",
        "first_flip_days": f"{int(round(oscillating['first_flip_median_days']))}",
        "still_flipping": (
            f"{stabilization['still_flipping_pct']:.1f}% "
            f"({stabilization['still_flipping_count']}/{stabilization['oscillating_lineages']})"
        ),
    }


table8_display = pd.DataFrame([
    {"Category": "Transition gap (median days)", "Sigma": "", "SSC": ""},
    {
        "Category": "CE → CE ; FPR → FPR",
        "Sigma": table8_by_dataset["Sigma"]["same_direction_gap"],
        "SSC": table8_by_dataset["SSC"]["same_direction_gap"],
    },
    {
        "Category": "CE → FPR ; FPR → CE",
        "Sigma": table8_by_dataset["Sigma"]["cross_direction_gap"],
        "SSC": table8_by_dataset["SSC"]["cross_direction_gap"],
    },
    {"Category": "Oscillating lineages", "Sigma": "", "SSC": ""},
    {
        "Category": "Time to first flip (median days)",
        "Sigma": table8_by_dataset["Sigma"]["first_flip_days"],
        "SSC": table8_by_dataset["SSC"]["first_flip_days"],
    },
    {
        "Category": "Still flipping at snapshot",
        "Sigma": table8_by_dataset["Sigma"]["still_flipping"],
        "SSC": table8_by_dataset["SSC"]["still_flipping"],
    },
])

display(table8_display)


Table T1. Pair-Level Revision Gap by Rationale Label


,dataset,label,n,median_days,p25_days,p75_days,mean_days,max_days
0,SSC,coverage_expansion,1586,18.37,4.00,57.38,51.32,979.40
1,SSC,false_positive_reduction,793,9.97,1.20,30.98,39.51,1103.51
2,SSC,mixed_tradeoff,505,11.58,1.42,35.79,40.90,764.18
3,SSC,insufficient_evidence,1528,9.19,1.27,35.79,46.39,764.18
4,Sigma,coverage_expansion,3289,26.94,3.66,101.38,70.47,637.13
5,Sigma,false_positive_reduction,2401,13.67,1.27,62.69,52.04,655.67
6,Sigma,mixed_tradeoff,952,17.66,1.26,83.80,69.49,542.07
7,Sigma,insufficient_evidence,1316,24.04,1.97,95.89,66.87,971.48


,Category,Sigma,SSC
0,Transition gap (median days),,
1,CE → CE ; FPR → FPR,125 ; 32,187 ; 110
2,CE → FPR ; FPR → CE,44 ; 74,21 ; 68
3,Oscillating lineages,,
4,Time to first flip (median days),144,51
5,Still flipping at snapshot,58.4% (201/344),74.2% (118/159)
